# ANNITIA Challenge — MASLD Survival Prediction

**Author:** YEVI Mawuli Peniel Samuel — IFRI-UAC (Benin)
**Challenge:** ANNITIA by TRUSTII / ICAN
**Objective:** Predict two survival endpoints from longitudinal NITs:
- `risk_hepatic_event` — risk of major hepatic event (MASH, decompensation, HCC)
- `risk_death` — all-cause mortality risk

**Score:** `0.7 × C-index_hepatic + 0.3 × C-index_death`

---

## Pipeline Overview

```
Wide CSV (1253 patients × 22 visits × 18 features)
        |
        ▼
Temporal Feature Engineering (162 features)
  • Per-feature statistics  : last, first, max, min, mean, std, slope, delta
  • Temporal acceleration   : slope_recent, accel (2nd half of visits)
  • Validated MASLD ratios  : FIB-4, AST/ALT, GGT/ALT, FibroScan×FibroTest
  • Missingness patterns    : obs_rate, n_features_last_visit
        |
        ▼
XGBoost Cox PH  (native survival:cox objective)
  • 5-fold stratified CV + early stopping
  • 5 seeds -> averaged predictions (bagging)
        |
        ▼
Risk scores [0, +∞)
```

**Complementary model:** K-Mamba SSM — sequential neural network (State Space Model)
trained on raw longitudinal trajectories. Implemented in pure C with end-to-end backpropagation.

## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import StratifiedKFold
from pathlib import Path

# ─── Data paths (auto-detected — update manually if needed) ──────────────────
_candidates_train = [
    'data/processed_wide/train.csv',
    'data/train.csv',
    '../data/train.csv',
    'train.csv',
]
_candidates_test = [
    'data/processed_wide/val_test.csv',
    'data/val_test.csv',
    'data/test.csv',
    '../data/val_test.csv',
    '../data/test.csv',
    'val_test.csv',
    'test.csv',
]

def _find_csv(candidates):
    for p in candidates:
        if Path(p).exists():
            return p
    raise FileNotFoundError(
        f"Data file not found. Tried: {candidates}\n"
        "Update TRAIN_CSV / TEST_CSV manually.")

TRAIN_CSV  = _find_csv(_candidates_train)
TEST_CSV   = _find_csv(_candidates_test)
OUTPUT_CSV = 'submission_1.csv'
MODEL_PATH = 'xgboost_annitia.json'
N_VISITS   = 22
N_FOLDS    = 5
SEEDS      = [42, 123, 777, 2024, 31415]

print(f'Train CSV : {TRAIN_CSV}')
print(f'Test  CSV : {TEST_CSV}')
print(f'XGBoost   : {xgb.__version__}')


## 2. Data Loading

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train : {df_train.shape[0]} patients, {df_train.shape[1]} columns')
print(f'Test  : {df_test.shape[0]} patients')

age_cols   = [f'Age_v{v}' for v in range(1, N_VISITS+1)]

event_hep  = df_train['evenements_hepatiques_majeurs'].fillna(0).astype(int).values
time_hep   = df_train['evenements_hepatiques_age_occur'].where(
                 df_train['evenements_hepatiques_majeurs'] == 1,
                 df_train[age_cols].max(axis=1)
             ).fillna(1e-3).values

event_dth  = df_train['death'].fillna(0).astype(int).values
time_dth   = df_train['death_age_occur'].where(
                 df_train['death'] == 1,
                 df_train[age_cols].max(axis=1)
             ).fillna(1e-3).values

print(f'\nHepatic events : {event_hep.sum()} ({100*event_hep.mean():.1f}%)')
print(f'Deaths         : {event_dth.sum()} ({100*event_dth.mean():.1f}%)')
print(f'Median follow-up : {np.median(df_train[age_cols].max(axis=1) - df_train["Age_v1"]):.1f} years')

## 3. Temporal Feature Engineering

Each patient has repeated measurements over up to 22 visits. For each dynamic feature
(BMI, ALT, AST, bilirubin, cholesterol, GGT, fasting glucose, platelets, triglycerides,
Aixplorer, FibroTest, FibroScan), we extract:

| Statistic | Description |
|-----------|-------------|
| `last`, `first` | Value at last / first visit |
| `max`, `min`, `mean`, `std` | Statistics across all visits |
| `slope` | Linear trend (OLS) over the full follow-up |
| `delta` | Absolute change: last − first |
| `slope_recent` | Trend over the second half of visits |
| `accel` | Acceleration: mean(2nd half) − mean(1st half) |
| `obs_rate` | Fraction of visits with a recorded measurement |

**Validated MASLD clinical ratios:**
- **FIB-4** = (age × AST) / (platelets × √ALT) — standard fibrosis marker
- **De Ritis** (AST/ALT) > 1 -> likely advanced fibrosis
- **GGT/ALT** — hepatocellular injury vs. cholestasis
- **FibroScan × FibroTest** — concordance of two non-invasive fibrosis measures

In [ ]:
# Feature engineering — self-contained implementation

DYN_PREFIXES = [
    'BMI_v', 'alt_v', 'ast_v', 'bilirubin_v', 'chol_v', 'ggt_v',
    'gluc_fast_v', 'plt_v', 'triglyc_v',
    'aixp_aix_result_BM_3_v', 'fibrotest_BM_2_v', 'fibs_stiffness_med_BM_1_v',
]
DYN_NAMES = [
    'BMI', 'ALT', 'AST', 'bilirubin', 'chol', 'GGT',
    'gluc_fast', 'plt', 'triglyc', 'Aixplorer', 'FibroTest', 'FibroScan',
]
STAT_COLS = ['gender', 'T2DM', 'Hypertension', 'Dyslipidaemia',
             'bariatric_surgery', 'bariatric_surgery_age']

def _slope(values):
    """OLS slope on non-NaN values."""
    mask = ~np.isnan(values)
    if mask.sum() < 2:
        return np.nan
    x = np.where(mask)[0].astype(float)
    y = values[mask]
    x -= x.mean()
    d = (x ** 2).sum()
    return float((x * y).sum() / d) if d > 0 else 0.0

def engineer_features(df):
    rows = []
    for _, row in df.iterrows():
        f = {}
        ages = np.array([
            float(row[f'Age_v{v}']) if f'Age_v{v}' in row.index and not pd.isna(row[f'Age_v{v}'])
            else np.nan for v in range(1, N_VISITS + 1)
        ])
        age_v1   = float(ages[0]) if not np.isnan(ages[0]) else 0.0
        vm       = ~np.isnan(ages)
        n_visits = int(vm.sum())
        last_age = float(ages[vm][-1]) if n_visits > 0 else age_v1
        f['n_visits']  = n_visits
        f['age_v1']    = age_v1
        f['follow_up'] = last_age - age_v1

        for prefix, name in zip(DYN_PREFIXES, DYN_NAMES):
            vals = np.array([
                float(row[f'{prefix}{v}'])
                if f'{prefix}{v}' in row.index and not pd.isna(row[f'{prefix}{v}'])
                else np.nan for v in range(1, N_VISITS + 1)
            ])
            valid = vals[~np.isnan(vals)]
            f[f'{name}_last']     = float(valid[-1])    if len(valid) > 0 else np.nan
            f[f'{name}_first']    = float(valid[0])     if len(valid) > 0 else np.nan
            f[f'{name}_max']      = float(valid.max())  if len(valid) > 0 else np.nan
            f[f'{name}_min']      = float(valid.min())  if len(valid) > 0 else np.nan
            f[f'{name}_mean']     = float(valid.mean()) if len(valid) > 0 else np.nan
            f[f'{name}_std']      = float(valid.std())  if len(valid) > 1 else 0.0
            f[f'{name}_slope']    = _slope(vals)
            f[f'{name}_n_obs']    = int(len(valid))
            f[f'{name}_delta']    = float(valid[-1] - valid[0]) if len(valid) >= 2 else np.nan
            f[f'{name}_obs_rate'] = len(valid) / N_VISITS

            vi = np.where(~np.isnan(vals))[0]
            if len(vi) >= 4:
                mid = len(vi) // 2
                first_half  = vals[vi[:mid]]
                second_half = vals[vi[mid:]]
                f[f'{name}_slope_recent'] = _slope(vals[vi[mid:]])
                f[f'{name}_accel']        = float(second_half.mean() - first_half.mean())
            else:
                f[f'{name}_slope_recent'] = np.nan
                f[f'{name}_accel']        = np.nan

        for col in STAT_COLS:
            f[col] = float(row[col]) if col in row.index and not pd.isna(row[col]) else np.nan

        alt   = f.get('ALT_last',       np.nan)
        ast   = f.get('AST_last',       np.nan)
        ggt   = f.get('GGT_last',       np.nan)
        plt_  = f.get('plt_last',       np.nan)
        bili  = f.get('bilirubin_last', np.nan)
        bmi   = f.get('BMI_last',       np.nan)
        fibro = f.get('FibroScan_last', np.nan)
        ftest = f.get('FibroTest_last', np.nan)

        f['AST_ALT_ratio']             = ast / alt  if (not any(np.isnan([ast, alt]))  and alt  != 0) else np.nan
        f['GGT_ALT_ratio']             = ggt / alt  if (not any(np.isnan([ggt, alt]))  and alt  != 0) else np.nan
        f['FIB4']                      = (age_v1 * ast) / (plt_ * np.sqrt(alt)) \
                                         if (not any(np.isnan([age_v1, ast, plt_, alt])) and plt_ > 0 and alt > 0) else np.nan
        f['FibroScan_x_FibroTest']     = fibro * ftest if not any(np.isnan([fibro, ftest])) else np.nan
        f['FibroScan_FibroTest_ratio'] = fibro / ftest if (not any(np.isnan([fibro, ftest])) and ftest != 0) else np.nan
        f['bili_x_AST']                = bili  * ast   if not any(np.isnan([bili,  ast]))  else np.nan
        f['fibro_x_age']               = fibro * age_v1 if not np.isnan(fibro)             else np.nan
        f['BMI_x_fibro']               = bmi   * fibro if not any(np.isnan([bmi,  fibro])) else np.nan
        f['n_features_last_visit']     = sum(
            1 for px in DYN_PREFIXES
            if f'{px}{n_visits}' in df.columns and not pd.isna(row.get(f'{px}{n_visits}', np.nan))
        ) if n_visits > 0 else 0

        rows.append(f)
    return pd.DataFrame(rows)

print('Running feature engineering...')
X_train = engineer_features(df_train)
X_test  = engineer_features(df_test)
print(f'Features extracted : {X_train.shape[1]}')
print(f'First features : {list(X_train.columns[:5])}')
print(f'Last features  : {list(X_train.columns[-5:])}')

## 4. XGBoost Cox PH Model

### Why XGBoost Cox?

| Criterion | RSF | Cox ElasticNet | **XGBoost Cox** |
|-----------|-----|----------------|-----------------|
| Censoring | [Good] | [Good] | [Good] native |
| Non-linearities | [Good] | [Poor] | [Good] |
| Feature interactions | [Good] | [Poor] | [Good] |
| Regularization | [Poor] | [Good] L1+L2 | [Good] L1+L2 |
| Missing values | [Poor] | [Poor] | [Good] native |
| Low EPV (1.3) | [Poor] overfit | [Good] | [Good] |

**`survival:cox` objective:** Cox Proportional Hazard model via gradient boosting.
Labels: `y = +time` for events, `y = −time` for censored observations.

### 5-fold Stratified Cross-Validation

Stratification on both hepatic event AND death simultaneously, ensuring balanced
rare-event distribution across folds (hepatic EPV ≈ 1.3).

In [ ]:
def c_index_numpy(risks, times, events):
    """Harrell C-index (numpy implementation, O(n^2))."""
    risks, times, events = np.asarray(risks), np.asarray(times), np.asarray(events)
    concordant = discordant = tied = 0
    for i in range(len(times)):
        if events[i] == 0:
            continue
        for j in range(len(times)):
            if i == j or times[j] <= times[i]:
                continue
            if risks[i] > risks[j]:
                concordant += 1
            elif risks[i] < risks[j]:
                discordant += 1
            else:
                tied += 1
    total = concordant + discordant + tied
    return (concordant + 0.5 * tied) / total if total > 0 else 0.5


XGB_PARAMS = dict(
    objective        = 'survival:cox',
    eval_metric      = 'cox-nloglik',
    tree_method      = 'hist',
    learning_rate    = 0.05,
    max_depth        = 4,
    min_child_weight = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    nthread          = -1,
    verbosity        = 0,
)

feat_names = X_train.columns.tolist()


def train_xgb_cv(X, events, times, seeds=SEEDS, n_folds=N_FOLDS):
    """K-fold stratifié multi-seed -> OOF + prédictions test."""
    oof_all, test_all = [], []
    for seed in seeds:
        skf_s = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_seed  = np.zeros(len(X))
        test_seed = np.zeros(len(X_test))
        y_cox = np.where(events == 1, times, -times).astype(np.float32)
        for tr_idx, va_idx in skf_s.split(X, events):
            X_tr, X_va = X.iloc[tr_idx].values, X.iloc[va_idx].values
            y_tr, y_va = y_cox[tr_idx], y_cox[va_idx]
            dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=feat_names)
            dval   = xgb.DMatrix(X_va, label=y_va, feature_names=feat_names)
            dtest  = xgb.DMatrix(X_test.values, feature_names=feat_names)
            model  = xgb.train(
                dict(**XGB_PARAMS, seed=seed),
                dtrain,
                num_boost_round=500,
                evals=[(dval, 'val')],
                early_stopping_rounds=50,
                verbose_eval=False,
            )
            oof_seed[va_idx]  = model.predict(dval)
            test_seed        += model.predict(dtest) / n_folds
        oof_all.append(oof_seed)
        test_all.append(test_seed)
    return np.mean(oof_all, axis=0), np.mean(test_all, axis=0)


print('Training XGBoost Cox PH — hepatic endpoint...')
oof_hep, test_hep = train_xgb_cv(X_train, event_hep, time_hep)
ci_hep = c_index_numpy(oof_hep, time_hep, event_hep)
print(f'  OOF C-index hepatic: {ci_hep:.4f}')

print('Training XGBoost Cox PH — death endpoint...')
oof_dth, test_dth = train_xgb_cv(X_train, event_dth, time_dth)
ci_dth = c_index_numpy(oof_dth, time_dth, event_dth)
print(f'  OOF C-index death:    {ci_dth:.4f}')

score_oof = 0.7 * ci_hep + 0.3 * ci_dth
print(f'\nOOF Score (0.7×hepatic + 0.3×death) = {score_oof:.4f}')

## 5. Results — Model Comparison

| Model | OOF C-index Hepatic | OOF C-index Death | OOF Score | Notes |
|-------|--------------------|--------------------|-----------|-------|
| Cox ElasticNet (baseline) | ~0.77 | ~0.72 | ~0.75 | Flat features, 5 active coefs |
| RSF (baseline) | ~0.91 | ~0.93 | ~0.91 | **Train score — clear overfit** |
| LightGBM binary + temporal weights | ~0.78 | ~0.64 | ~0.73 | Suboptimal objective |
| K-Mamba SSM (19K params, 100 epochs) | 0.824 | 0.882 | 0.842 | Temporal trajectories |
| **XGBoost Cox multi-seed** | **0.885** | **0.876** | **0.882** | **-> Final submission** |

## 6. Feature Importance

Importance measured by **total gain** contributed by each feature across all tree splits.

In [ ]:
# Train a full-data model for feature importance
y_cox_hep   = np.where(event_hep == 1, time_hep, -time_hep).astype(np.float32)
dtrain_full = xgb.DMatrix(X_train.values.astype(np.float32), label=y_cox_hep, feature_names=feat_names)
model_full  = xgb.train(dict(**XGB_PARAMS, seed=42), dtrain_full,
                        num_boost_round=300, verbose_eval=False)

imp = pd.Series(model_full.get_score(importance_type='gain')).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if any(k in f for k in ['Fibro', 'FIB4', 'AST', 'bili'])
          else '#1f77b4' for f in imp.index]
ax.barh(imp.index[::-1], imp.values[::-1], color=colors[::-1])
ax.set_xlabel('Gain (XGBoost feature importance)')
ax.set_title('Top 20 Features — Hepatic Event (XGBoost Cox)', fontsize=13)
red_patch  = mpatches.Patch(color='#d62728', label='Fibrosis / hepatic injury markers')
blue_patch = mpatches.Patch(color='#1f77b4', label='Other temporal features')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.savefig('feature_importance_hepatic.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClinical interpretation of top features:')
interpretations = {
    'FibroScan_max'   : 'Maximum liver stiffness -> advanced fibrosis',
    'AST_max'         : 'Peak AST -> necroinflammatory activity',
    'FibroScan_mean'  : 'Mean liver stiffness -> global fibrotic state',
    'bili_x_AST'      : 'Bilirubin × AST -> combined hepatic insufficiency marker',
    'FibroScan_accel' : 'FibroScan acceleration (2nd half) -> fibrosis progression',
    'FIB4'            : 'FIB-4 = (age×AST)/(plt×√ALT) -> clinical standard for fibrosis',
    'GGT_std'         : 'GGT variability -> cholestasis/hepatocellular instability',
}
for feat, interp in list(interpretations.items())[:7]:
    print(f'  {feat:30s} -> {interp}')

## 7. K-Mamba SSM Architecture (complementary temporal model)

In parallel with the XGBoost pipeline, we developed a **State Space Model (SSM)**
inspired by the Mamba architecture, implemented in **pure C** with end-to-end backpropagation.

### Why a SSM for longitudinal data?

MASLD NITs are **irregular time series**: each patient has between 1 and 22 visits
with variable intervals. A SSM naturally handles:
- **Temporal ordering** of measurements
- **Missing data** via a visit mask
- **Long-range dependencies** between distant visits

```
Patient [T visits, 18 features]
    ↓  W_feat [18 -> 64]  +  W_time [1 -> 64]
hidden [T, 64]
    ↓  MambaBlock #1  (MIMO rank=4, state=16)
    ↓  MambaBlock #2
last-valid-timestep pooling
    ↓  W_hepatic [64 -> 1]    W_death [64 -> 1]
risk_hepatic,  risk_death
```

**Survival loss:**
`L = α × ranking_loss + (1−α) × cox_loss`
with `α_hepatic = 0.2` (more ranking loss for rare events at 3.8%)
and `α_death = 0.5`.

**SSM results (5-fold OOF, 100 epochs, mimo_rank=4):**
C-index hepatic = 0.824, C-index death = 0.882, **score = 0.842**

**Correlation with XGBoost on test set:**
- Hepatic: **0.142** (near-independent)
- Death: **−0.014** (independent)

The SSM captures complementary temporal patterns (fibrosis progression acceleration
visible in sequences), with further potential at higher epoch counts.

## 8. Ensemble — XGBoost + K-Mamba SSM

The SSM was trained externally (pure C, requires `git clone --recursive`).
Predictions on the 423 test patients are provided as pre-computed `.npy` files
so the ensemble can be verified without recompiling the C library.

**Rank-average ensemble:** normalized rank scores are averaged with weights
`w_ssm=0.3, w_xgb=0.7` — chosen based on relative OOF performance
(SSM OOF: 0.842, XGBoost OOF: 0.882).

**Correlation between models on test set:**
- Hepatic: **0.142** — near-independent
- Death: **−0.014** — independent

Near-zero correlation confirms orthogonal signal capture between the two models.

In [ ]:
def _to_rank(scores):
    """Normalize scores to ranks in [0, 1]."""
    n = len(scores)
    order = np.argsort(scores)
    ranks = np.empty(n)
    ranks[order] = np.arange(n)
    return ranks / (n - 1)

# Load pre-computed SSM predictions (trained externally — see repo for C source)
_ssm_hep_path = Path('predictions/test_ssm_hep.npy')
_ssm_dth_path = Path('predictions/test_ssm_dth.npy')

if _ssm_hep_path.exists() and _ssm_dth_path.exists():
    ssm_hep = np.load(_ssm_hep_path)
    ssm_dth = np.load(_ssm_dth_path)

    w_ssm, w_xgb = 0.3, 0.7
    ens_hep = w_ssm * _to_rank(ssm_hep) + w_xgb * _to_rank(test_hep)
    ens_dth = w_ssm * _to_rank(ssm_dth) + w_xgb * _to_rank(test_dth)

    corr_hep = float(np.corrcoef(_to_rank(ssm_hep), _to_rank(test_hep))[0, 1])
    corr_dth = float(np.corrcoef(_to_rank(ssm_dth), _to_rank(test_dth))[0, 1])

    print(f'SSM predictions loaded : {len(ssm_hep)} patients')
    print(f'Correlation SSM / XGBoost — hepatic : {corr_hep:.3f}')
    print(f'Correlation SSM / XGBoost — death   : {corr_dth:.3f}')

    sub_ensemble = pd.DataFrame({
        'trustii_id'        : trustii_ids,
        'risk_hepatic_event': ens_hep,
        'risk_death'        : ens_dth,
    })
    sub_ensemble.to_csv('submission_2.csv', index=False)
    print(f'\nEnsemble submission saved : submission_2.csv')
    print(sub_ensemble.head(5).to_string(index=False))
else:
    print('SSM .npy files not found — skipping ensemble (XGBoost only submission used)')

In [ ]:
# ─── Read test IDs ───────────────────────────────────────────────────────────
# The source column may be 'trustii_id' or 'trustii-id' depending on data version.
_id_col = 'trustii-id' if 'trustii-id' in df_test.columns else 'trustii_id'
trustii_ids = df_test[_id_col].values

submission = pd.DataFrame({
    'trustii-id'        : trustii_ids,
    'risk_hepatic_event': test_hep,
    'risk_death'        : test_dth,
})

submission.to_csv(OUTPUT_CSV, index=False)

print(f'Submission saved: {OUTPUT_CSV}')
print(f'Patients: {len(submission)}')
print(f'\nPreview:')
print(submission.head(10).to_string(index=False))
print(f'\nPrediction statistics:')
print(submission[['risk_hepatic_event', 'risk_death']].describe().round(4).to_string())


## 10. Model Save / Load & API Submission

**Required by Trustii**: the notebook must include code to save and load the model. The Trustii team must be able to reproduce predictions by running this notebook end-to-end.

In [ ]:
# ─── Model Save / Load — MANDATORY for Trustii final submission ───────────────
# Train a final XGBoost model on the FULL training set (all 5 seeds averaged)
# and save it so Trustii can reproduce results without re-running 5-fold CV.

print('Training final XGBoost model on full training data...')

y_cox_hep_full = np.where(event_hep == 1, time_hep, -time_hep).astype(np.float32)
y_cox_dth_full = np.where(event_dth == 1, time_dth, -time_dth).astype(np.float32)
feat_names_full = X_train.columns.tolist()

# Average predictions from multiple seeds (same protocol as CV)
final_preds_hep = np.zeros(len(X_test))
final_preds_dth = np.zeros(len(X_test))

for seed in SEEDS:
    dtrain_hep = xgb.DMatrix(X_train.values, label=y_cox_hep_full, feature_names=feat_names_full)
    dtest_full  = xgb.DMatrix(X_test.values, feature_names=feat_names_full)

    model_hep_full = xgb.train(
        dict(**XGB_PARAMS, seed=seed),
        dtrain_hep,
        num_boost_round=350,   # fixed rounds (no val set — use best from CV)
        verbose_eval=False,
    )
    final_preds_hep += model_hep_full.predict(dtest_full) / len(SEEDS)

    dtrain_dth = xgb.DMatrix(X_train.values, label=y_cox_dth_full, feature_names=feat_names_full)
    model_dth_full = xgb.train(
        dict(**XGB_PARAMS, seed=seed),
        dtrain_dth,
        num_boost_round=350,
        verbose_eval=False,
    )
    final_preds_dth += model_dth_full.predict(dtest_full) / len(SEEDS)

# Save last-seed models as representative (all seeds share same architecture)
model_hep_full.save_model('model_hepatic.json')
model_dth_full.save_model('model_death.json')
print(f'Models saved: model_hepatic.json, model_death.json')

# ─── Load and verify ──────────────────────────────────────────────────────────
model_hep_loaded = xgb.Booster()
model_hep_loaded.load_model('model_hepatic.json')
model_dth_loaded = xgb.Booster()
model_dth_loaded.load_model('model_death.json')

dtest_check = xgb.DMatrix(X_test.values, feature_names=feat_names_full)
preds_hep_check = model_hep_loaded.predict(dtest_check)
preds_dth_check = model_dth_loaded.predict(dtest_check)

print(f'Load verification OK — hepatic pred range: [{preds_hep_check.min():.4f}, {preds_hep_check.max():.4f}]')
print(f'Load verification OK — death   pred range: [{preds_dth_check.min():.4f}, {preds_dth_check.max():.4f}]')
print('Model save/load: [Good]')


In [ ]:
# ─── Submission via Trustii API (optional — can also upload manually) ─────────
import requests

def send_submission_via_api(csv_file_path, ipynb_file_path, token, challenge_id):
    """Submit predictions and notebook to Trustii.io via API."""
    endpoint_url = f'https://api.trustii.io/api/ds/notebook/datasets/{challenge_id}/prediction'
    with open(csv_file_path, 'rb') as csv_file:
        csv_data = csv_file.read()
    with open(ipynb_file_path, 'rb') as ipynb_file:
        ipynb_data = ipynb_file.read()
    headers = {'Authorization': f'UserApiKey {token}'}
    files = {
        'file': (csv_file_path, csv_data, 'text/csv'),
        'notebook': (ipynb_file_path, ipynb_data, 'application/json'),
    }
    response = requests.post(endpoint_url, headers=headers, files=files)
    if response.status_code == 200:
        print('Submission successful:', response.json())
    else:
        print(f'Submission failed ({response.status_code}):', response.text)
    return response

# To submit, fill in your token and challenge_id from https://app.trustii.io
# TRUSTII_TOKEN      = 'your-token-here'
# TRUSTII_CHALLENGE  = 'your-challenge-id-here'
# send_submission_via_api(OUTPUT_CSV, 'notebook_annitia.ipynb', TRUSTII_TOKEN, TRUSTII_CHALLENGE)

print('API submission cell ready.')
print('Fill in TRUSTII_TOKEN and TRUSTII_CHALLENGE, then uncomment the last line.')


## 9. Conclusion

### Final Pipeline

**XGBoost Cox multi-seed** — enriched temporal feature engineering:
- 162 features extracted from longitudinal trajectories (vs. 35–44 flat features in baseline)
- Native `survival:cox` objective handles censored data correctly
- 5-fold stratified CV + 5 seeds -> stable predictions (bagging)
- **Estimated OOF score: 0.882**

### Key Findings

1. **FibroScan is the dominant marker** — maximum liver stiffness and its acceleration
   predict better than point-in-time values. This confirms the importance of **longitudinal follow-up**.

2. **FIB-4 index** (age×AST)/(plt×√ALT) — clinical standard independently recovered as a top feature,
   validating the clinical coherence of the model.

3. **Temporal acceleration** (change between 1st and 2nd half of follow-up) captures
   fibrosis progression dynamics that classical statistics (mean, std) miss entirely.

4. **K-Mamba SSM** (pure C sequential model) reaches 0.842 OOF at 100 epochs —
   complementary signal with near-zero correlation to XGBoost (0.14 hepatic, −0.01 death),
   suggesting orthogonal information capture from raw trajectories.

5. **Performance ceiling analysis** via Gaussian Process optimizer shows that reaching
   Score ≥ 0.95 requires extending the model space — specifically 2D scanning
   over the patient matrix [visits × features] to automatically learn cross-feature interactions.